Evaluate the performance of VAGO on the manually labelled subset

In [4]:
import sys
import os

base_dir = (
    os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
)
sys.path.insert(0, os.path.abspath(os.path.join(base_dir, "..")))

import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from config import VAGUENESS_TYPES

In [5]:
df = pd.read_csv("../../output/analyzed/manual_vagueness_analyzed_updated.csv")
# Filter out not green claims
df = df[df["Vagueness Label"] != 2].copy()

In [6]:
def count_vague_words(row):
    total = 0
    for col in VAGUENESS_TYPES:
        val = str(row[col]).strip()
        if val and val.lower() != "nan":
            # Split on comma+space to count multiple words in one cell
            words = [w.strip() for w in val.split(",") if w.strip()]
            total += len(words)
    return total

In [7]:
df["vague_word_count"] = df.apply(count_vague_words, axis=1)

y_true = df["Vagueness Label"].astype(int)

print(
    f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Predicted Vague':<16} {'Support'}"
)

for threshold in range(1, df["vague_word_count"].max() + 2):
    y_pred = (df["vague_word_count"] >= threshold).astype(int)

    if y_pred.sum() == 0:
        print(
            f"{threshold:<12} {'N/A':<12} {'0.00':<12} {'N/A':<12} {0:<16} {y_true.sum()}"
        )
        break

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(
        f"{threshold:<12} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f} {y_pred.sum():<16} {y_true.sum()}"
    )


Threshold    Precision    Recall       F1           Predicted Vague  Support
1            0.5962       0.8000       0.6832       577              430
2            0.6205       0.6465       0.6333       448              430
3            0.6801       0.4302       0.5271       272              430
4            0.6909       0.2651       0.3832       165              430
5            0.7108       0.1372       0.2300       83               430
6            0.7727       0.0791       0.1435       44               430
7            0.9231       0.0279       0.0542       13               430
8            0.8333       0.0116       0.0229       6                430
9            1.0000       0.0023       0.0046       1                430
10           1.0000       0.0023       0.0046       1                430
11           1.0000       0.0023       0.0046       1                430
12           N/A          0.00         N/A          0                430


Basic vagueness summary statistics (on subset)

In [15]:
_PERIOD = {True: "pre-legislation", False: "post-legislation"}
_METRIC = {
    "vague_claims": "Vague Green Claims",
    "precise_claims": "Precise Green Claims",
    "total_rows": "Total Green Claims",
    "normalized_vague": "Normalized (Vague)",
    "normalized_precise": "Normalized (Precise)",
}

_COL_ORDER = [f"{m} ({p})" for m in _METRIC.values() for p in _PERIOD.values()]

def summary_table(grouped):
    """
    For a (possibly grouped) DataFrame, compute:
      - Total Green Claims      : all rows (Vagueness Label 0 or 1)
      - Vague Green Claims      : rows where Vagueness Label == 1
      - Precise Green Claims    : rows where Vagueness Label == 0
      - Normalized (Vague)      : Vague / Total
      - Normalized (Precise)    : Precise / Total
    """
    return grouped.agg(
        **{
            "Total Green Claims": ("Vagueness Label", "count"),
            "Vague Green Claims": (
                "Vagueness Label",
                "sum",
            ),  # 1=vague, 0=not -> sum = vague count
        }
    ).assign(
        **{
            "Precise Green Claims": lambda x: x["Total Green Claims"]
            - x["Vague Green Claims"]
        },
        **{
            "Normalized (Vague)": lambda x: (
                x["Vague Green Claims"] / x["Total Green Claims"]
            ).round(4)
        },
        **{
            "Normalized (Precise)": lambda x: (
                x["Precise Green Claims"] / x["Total Green Claims"]
            ).round(4)
        },
    )

In [ ]:
wayback_table = summary_table(df.groupby("isWayback"))
wayback_table.index = wayback_table.index.map(_PERIOD)

print("=== Before vs. After Bill C-59 ===")
print(wayback_table.to_string())

org_table = summary_table(df.groupby("Organization")).sort_values(
    "Vague Green Claims", ascending=False
)

print("\n=== By Organization ===")
print(org_table.to_string())

org_wayback_pivot = (
    df.groupby(["Organization", "isWayback"])
    .agg(
        vague_claims=("Vagueness Label", "sum"),
        total_rows=("Vagueness Label", "count"),
    )
    .assign(
        precise_claims=lambda x: x["total_rows"] - x["vague_claims"],
        normalized_vague=lambda x: (x["vague_claims"] / x["total_rows"]).round(4),
        normalized_precise=lambda x: (x["precise_claims"] / x["total_rows"]).round(4),
    )
    .unstack("isWayback")
)

org_wayback_pivot.columns = [
    f"{_METRIC[m]} ({_PERIOD[wb]})" for m, wb in org_wayback_pivot.columns
]
org_wayback_pivot = org_wayback_pivot[_COL_ORDER]

print(
    "\n=== Vague Green Claims by Organization (pre-legislation vs post-legislation) ==="
)
print(org_wayback_pivot.to_string())

=== Before vs. After Bill C-59 ===
                  Total Green Claims  Vague Green Claims  Precise Green Claims  Normalized (Vague)  Normalized (Precise)
isWayback                                                                                                               
post-legislation                 201                 129                    72              0.6418                0.3582
pre-legislation                  500                 301                   199              0.6020                0.3980

=== By Organization ===
                            Total Green Claims  Vague Green Claims  Precise Green Claims  Normalized (Vague)  Normalized (Precise)
Organization                                                                                                                      
Enbridge                                   277                 164                   113              0.5921                0.4079
Suncor Energy                              139                 